# Basics of Mobile Robotics - Project Report
### Prof. Francesco Mondada
###  EPFL - Fall 2025
###  Group 14
- <b> Jules VILLARET </b>(Global Navigation)
- <b> Tim KOBLER </b>(Filtering)
- <b> Amir LAHLOU </b>(Vision)
- <b> Ana SCHWABEDAL </b>(Local Navigation & Motion Control)

## Table of Contents
0. [Introduction](#introduction)
1. [Environment Design](#environment-design)
2. [Individual Modules](#individual-modules)
    - [Vision](#vision)
    - [Global Navigation](#global-navigation)
    - [Filtering](#filtering)
    - [Local Navigation](#local-navigation)
    - [Motion Control](#motion-control)
3. [Vision](#vision)
4. [Full System Integration](#full-system-integration)
5. [Conclusion](#conclusion)
6. [References](#references)


## Introduction
Imagine a small robot casually exploring the surface of the Moon. A dust storm is approaching, and the robot must autonomously navigate through craters and rock to return safely to its lander. To succeed, it must perceive its surroundings, estimate its position despite uncertainty, plan a safe route, and react instantly to unexpected obstacles.

In this project, we recreate this scenario using the Thymio robot. Our goal is to design a complete navigation system that allows the robot to move reliably through a predefined map, reach a target placed anywhere in the environment and avoid both known and unexpected obstacles. The Thymio relies on an overhead camera for global pose estimation, an Extended Kalman Filter to correct odometry drift, and proximity sensors for reactive obstacle avoidance.

This report presents the main design choices and algorithms used to achieve these tasks. It follows the structure recommended in the MICRO-452 Basics of Mobile Robotics course taught by Prof. Francesco Mondada, and details the different modules of our system: computer vision, global navigation, filtering, local navigation and motion control.

## Global Navigation

### Objective

The role of the global navigation module is to compute a collision-free path from the robot’s current position to a target that can be placed anywhere on the map. In our setup, the environment consists two polygonal obstacles. The Thymio should avoid these obstacles without using its proximity sensors, meaning the global planner must ensure that the computed path is valid only based on the map.

To achieve this, we implemented a visibility-graph based global planner combined with Dijkstra’s shortest-path algorithm. The following libraries are required for numerical operations, graph computation, geometric calculations, and visualization:

In [ ]:
import numpy as np
import heapq
from math import sqrt
import constants as cst
import matplotlib.pyplot as plt

class GlobalNav:

    def __init__(self, arena_width=1189, arena_height=841):
        self.width = arena_width
        self.height = arena_height
        self.nodes = []
        self.edges = {}

### Why a Visibility Graph?

A visibility graph is a classical solution for motion planning in polygonal environments. It has several advantages relevant to our project:
- It guarantees optimal paths (shortest Euclidean distance).
- It works well with polygonal obstacles.
- It produces smooth global paths.
- It is simple enough to implement and debug, aligning with the course recommendation to keep the project as simple as possible.

The idea is to connect all obstacle vertices, the start, and the goal with straight segments whenever the segment is not obstructed by any polygon.

### Obstacle Inflation

One practical concern is that the Thymio has a non-zero radius (~55 mm). A path that technically “touches” an obstacle vertex could still lead to a collision. To avoid this, we inflate each obstacle by a constant inflation radius `INFLATION_R` of 140 mm :
- Compute each polygon’s centroid.
- Move every vertex outward along the direction of the centroid-to-vertex vector.

This ensures that:
- The visibility graph is computed around slightly “larger” obstacles.
- Any path found is safe for the physical robot.

This step is essential for creating a safe clearance around obstacles without requiring complex configuration-space geometry. You can find below the fucntion `inflate_polygon()`, who takes the list of the polygon vertices as input and outpouts the inflated vertices. 

In [2]:
    def inflate_polygon(self, poly): #inflate a polygon by moving each corner away from its centroid
            cx = np.mean([p[0] for p in poly]) #finfing centroid
            cy = np.mean([p[1] for p in poly])

            inflated = []
            for (x, y) in poly:
                vx = x - cx #dist from centroid to corner
                vy = y - cy
                L = sqrt(vx * vx + vy * vy) + 1e-6 #avoid div by 0, when center is on a point(like a 3 point line)
                scale = (L + cst.INFLATION_R) / L #scale factor to push vertex outward by inflation radius
                inflated.append((cx + vx * scale, cy + vy * scale)) #apply scaling to get inflated vertex
            return inflated

### Checking Segment Visibility
Two nodes A and B are connected by an edge only if segment AB is fully visible, meaning :
1. AB does not intersect any obstacle edge.
2. The midpoint of AB is not inside any inflated polygon (important when A or B lies directly on the obstacle boundary).

To determine whether two points were mutually visible, we relied on a few standard geometric tests such as checking segment intersections and detecting whether a point lies inside a polygon. These are classical tools used in computational geometry and widely applied in robot navigation.

- `ccw()` : Determines if three points are counter-clockwise, a prerequisite for segment intersection.
- `segments_intersect()` : Two segments AB and CD intersect if C and D lie on different sides of AB and A and B lie on different sides of CD.
- `segment_hits_polygon()` : Checks if the segment AB intersects any edges of the selected polygon.
- `point_in_polygon()` (ray casting) : Used to detect cases where a segment passes through a polygon without intersecting its edges.

These geometric tests ensure the correctness of the visibility graph in environments with arbitrary convex or concave shapes.

In [3]:
    
    def ccw(self, A, B, C): #are the points in counter-clockwise order? return True if > 0, False for cw < 0 and colinear = 0
        return (C[1] - A[1]) * (B[0] - A[0]) - (B[1] - A[1]) * (C[0] - A[0]) > 0 #https://www.geeksforgeeks.org/dsa/orientation-3-ordered-points/

    def segments_intersect(self, p1, p2, p3, p4): #return True if they cross
        return (self.ccw(p1, p3, p4) != self.ccw(p2, p3, p4)) and (self.ccw(p1, p2, p3) != self.ccw(p1, p2, p4)) #https://www.geeksforgeeks.org/dsa/check-if-two-given-line-segments-intersect/
        #two segments AB and CD intersect if and only if: C and D are on different sides of AB and A and B are on different sides of CD

    def segment_hits_polygon(self, A, B, poly): #true if segment AB intersects any edge of polygon poly
        n = len(poly)
        for i in range(n):
            C = poly[i]
            D = poly[(i + 1) % n] #pair of edges, loops around

            #if AB shares an endpoint with the edge CD -> allowed
            if A == C or A == D or B == C or B == D:
                continue #skip next if

            #check edge intersection
            if self.segments_intersect(A, B, C, D):
                return True  
        return False

    def point_in_polygon(self, point, poly):
        x, y = point
        inside = False
        n = len(poly)

        for i in range(n):
            x1, y1 = poly[i]
            x2, y2 = poly[(i + 1) % n]

            #cross product == 0 -> collinear
            cross = (x - x1) * (y2 - y1) - (y - y1) * (x2 - x1)
            if abs(cross) < cst.EPSILON: #accept tiny tolerance because never 0
                # dot product <= 0 -> point is between endpoints
                dot = (x - x1) * (x - x2) + (y - y1) * (y - y2)
                if dot <= cst.EPSILON:
                    return False  #on edge = considered outside

            #Ray casting : check if the horizontal ray crosses edge (x1,y1)-(x2,y2)
            if (y1 > y) != (y2 > y): #y is between y1 and y2; compute intersection x
                x_intersect = (x2 - x1) * (y - y1) / (y2 - y1) + x1
                if x < x_intersect:
                    inside = not inside
        return inside #True = inside, False = outside

    def segment_visible(self, A, B, polygon_list):
        mid = ((A[0] + B[0]) / 2.0, (A[1] + B[1]) / 2.0)

        for poly in polygon_list:
            if self.segment_hits_polygon(A, B, poly): #AB must not intersect any polygon edge
                return False
            if self.point_in_polygon(mid, poly): #midpoint inside polygon (for segments between the polygon's points)
                return False
        return True

### Building the Visibility Graph

The construction follows these steps:
1. Add start and goal points.
2. Add all inflated obstacle vertices.
3. For every pair of nodes (A, B):
    - Check whether segment AB is visible.
    - If yes, add an undirected edge with weight equal to the Euclidean distance in `edges`.

This produces:
- A list of `nodes` (coordinates in mm),
- A dictionary of `edges`, where each key is a node index and its value is a list of neighbors with associated distances.

This representation is suitable for all classical graph search algorithms.

In [ ]:
    def build_visibility_graph(self, start, goal, obstacles): #build the visibility graph on inflated obstacles
        """
        Inputs:
            start, goal: (x, y) points in mm
            obstacles: list of polygons [[(x,y),...], [...]]
        Returns:
            nodes: list of all points (start, goal, obstacle vertices)
            edges: dict {idx: [(neighbor_idx, distance), ...]}
        """
        inflated = [self.inflate_polygon(poly) for poly in obstacles] #inflate obstacles

        nodes = [start, goal] #idx 0 = start, 1 = goal
        for poly in inflated: #collect all nodes
            nodes.extend(poly)

        #build edges between all visible pairs
        N = len(nodes)
        edges = {i: [] for i in range(N)}

        for i in range(N):
            for j in range(i + 1, N):
                A = nodes[i]
                B = nodes[j]

                if self.segment_visible(A, B, inflated):
                    d = self.dist(A, B)
                    edges[i].append((j, d))
                    edges[j].append((i, d)) #adds to list with [(point, distance),...]

        return nodes, edges

### Shortest Path Computation

Once the visibility graph is constructed, we compute the shortest path using Dijkstra’s algorithm, implemented using a priority queue (<b>heapq</b>):
- Initialize with cost(start) = 0
- Pop the node with minimal accumulated distance
- Update neighbors if a shorter path is found
- Continue until reaching the goal

We then reconstruct the path using the `parent` dictionary.

Why Dijkstra?
- All edge weights are positive → Dijkstra is optimal.
- Faster and simpler than A*.
- Meets the course requirement of maintaining simplicity.

The final output is a list of nodes going from start to goal.

In [ ]:
    def dijkstra(self, edges, start_idx, goal_idx): #help from Exercise 5 part 1
        pq = [(0.0, start_idx)] #(cost_so_far, node_index), priority queue implemented with heapq
        cost = {start_idx: 0.0} #dictionary: node -> best known cost
        parent = {start_idx: None} #keeps track of how we reached each node

        while pq:
            c, u = heapq.heappop(pq) #pops the element with smallest cost, u=node and c=cost
            if u == goal_idx:
                break

            for v, w in edges[u]: #explore neighbors of u
                newc = c + w #total cost to reach v via u
                if v not in cost or newc < cost[v]: #update if first time visiting v or found better path
                    cost[v] = newc
                    parent[v] = u
                    heapq.heappush(pq, (newc, v))

        if goal_idx not in parent: #if goal has no parent, no path exists
            return None  # no path

        #reconstruct path, go backwards using the parent dictionary
        path = []
        cur = goal_idx
        while cur is not None:
            path.append(cur)
            cur = parent[cur]
        return path[::-1] #returns list of node indices for the shortest path.

### Path Output
The function `find_optimal_path()` returns:
- A list of waypoints from the robot’s current pose to the goal.
- These waypoints are then fed into the motion controller, which tracks the path segment by segment.

If no valid path exists, the function safely reports that no solution was found.

In [ ]:
    def find_optimal_path(self, thymio, goal, obstacles):
        """
        Inputs:
            thymio: (x, y, theta) in mm, rad
            goal: (x, y) in mm
            obstacles: list of polygons from vision
        Returns:
            path_world: list of (x, y) waypoints in mm.
        """
        start_pt = [thymio[0], thymio[1]]
        goal_pt = [goal[0], goal[1]]

        self.nodes, self.edges = self.build_visibility_graph(start_pt, goal_pt, obstacles)

        path_idx = self.dijkstra(self.edges, start_idx=0, goal_idx=1) #goal idx = 1 bc of build_visibility
        if path_idx is None:
            print("GlobalNav (visibility): No path found!")
            return []

        path_world = [self.nodes[i] for i in path_idx] #convert node indices to (x,y) coordinates
        return path_world


### Goal Reached?

To determine whether the Thymio has reached the goal, we use a simple distance-based criterion. Since the robot is not a point, we define a tolerance region around the goal `GOAL_TOL = 20`. If the robot’s current position lies within this radius, we consider the goal reached and stop global navigation. This is checked using the Euclidean distance.

In [4]:
    def is_goal_reached(self, current_pos, goal_pos): #tol = 70mm bc R of the Thymio is ~55, so a bit more 
        return self.dist(current_pos, goal_pos) < cst.GOAL_TOL
    

### Map Visualization
To help with debugging and for presentation purposes, we added a plotting function that displays:
- Original obstacles
- Inflated obstacles (implicitly through edges)
- All visibility edges
- All nodes
- The optimal path highlighted in red
- Start and goal points

The axes are scaled to match the arena dimensions (1189×841 mm) and oriented so that the origin matches our coordinate convention.

This visualization was extremely helpful to verify:
- Whether obstacle inflation was correct
- Whether occluded edges were properly removed
- Whether Dijkstra returned the expected path

It also provides a clear illustration for the final demo.

In [ ]:
# to add ![visibility_graph.png](attachment:visibility_graph.png)

### References

For the integration of this part, we used the listed references to guide us more into our work. However, note that the full code was all written by ourselves, taking into account the information provided by the following referenced websites:
- https://www.geeksforgeeks.org/dsa/orientation-3-ordered-points/
- https://www.geeksforgeeks.org/dsa/check-if-two-given-line-segments-intersect/
- https://chatgpt.com
- Exercise session in class